# 🚀 Zalo AI Challenge 2026: DataMate — SFT LoRA Training & Pilot Evaluation (Qwen2.5-Coder-7B)

> **Mục tiêu:** Fine-tune (SFT) mô hình `Qwen/Qwen2.5-Coder-7B-Instruct` bằng kỹ thuật LoRA trên tập **91 mẫu code chuẩn đã được verify** (`result/sft_train_ready.jsonl`), sau đó đánh giá ngay trên tập 30 câu Pilot để kiểm tra độ cải thiện Penalized Accuracy (PA).
> 
> **Pipeline gồm 8 bước:**
> 1. Kiểm tra phần cứng GPU (A100 / L4 / V100 / T4).
> 2. Kết nối Google Drive & Thiết lập workspace tại SSD cục bộ `/content/datamate`.
> 3. Cài đặt các thư viện cần thiết (`peft`, `transformers`, `trl`, `accelerate`, `datasets`).
> 4. Xác nhận Dataset SFT chuẩn hóa có đủ 91 mẫu kèm Recency Anchor.
> 5. Huấn luyện LoRA Adapter với Completion-Only Loss (chỉ tính loss trên code assistant).
> 6. Kiểm tra các trọng số LoRA Adapter đã lưu.
> 7. Chạy đánh giá Pilot 30 câu với LoRA Adapter vừa train và tính điểm PA mới.
> 8. Đồng bộ LoRA adapter, checkpoints và kết quả về Google Drive.

## Bước 1: Kiểm tra phần cứng & CUDA
Xác nhận môi trường đang kết nối GPU A100 (khuyến nghị cho bf16) hoặc L4/T4.

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
    print('bfloat16 supported:', torch.cuda.is_bf16_supported())

## Bước 2: Kết nối Google Drive & Thiết lập Workspace
Mount Google Drive và đồng bộ mã nguồn sang ổ SSD cục bộ `/content/datamate` để đạt tốc độ I/O nhanh nhất.

In [ ]:
import os, shutil

# Tự động bypass: Nếu Drive đã mount rồi thì KHÔNG gọi lại để không bị hiện popup phiền phức
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('✅ Google Drive đã kết nối sẵn — Đã bypass thành công, không cần bấm xác nhận lại!')

gdrive_project = '/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'
local_work = '/content/datamate'

assert os.path.exists(gdrive_project), f'Không tìm thấy thư mục dự án trên Drive: {gdrive_project}'

# Dùng rsync để copy nhanh và bỏ qua các file broken symlink của wandb / git / tmp
os.makedirs(local_work, exist_ok=True)
!rsync -a --exclude='.git' --exclude='tmp' --exclude='wandb' --exclude='__pycache__' --exclude='.ipynb_checkpoints' "$gdrive_project/" "$local_work/"

os.chdir(local_work)
print(f'✅ Đã chuyển workspace làm việc sang: {os.getcwd()}')
!ls -la


## Bước 3: Cài đặt Dependencies
Cài đặt `peft`, `transformers`, `accelerate`, `datasets` và các thư viện xử lý dữ liệu.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'peft', 'trl', 'accelerate', 'bitsandbytes',
    'datasets', 'python-calamine', 'pyyaml', 'tqdm', 'pytest',
], check=True)
# Plain BF16/FP16 LoRA does not use Colab's incompatible optional torchao.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

# Check real LoRA injection in a fresh process before loading the 7B weights.
subprocess.run([sys.executable, '-c', """
import torch, transformers, peft
from torch import nn
from peft import LoraConfig, get_peft_model
print('torch:', torch.__version__, 'transformers:', transformers.__version__, 'peft:', peft.__version__)
model = get_peft_model(nn.Sequential(nn.Linear(4, 4)), LoraConfig(target_modules=['0'], r=2))
assert any('lora_' in name and parameter.requires_grad for name, parameter in model.named_parameters())
print('LoRA dependency check passed')
"""], check=True)


## Bước 4: Kiểm tra Dataset SFT đã chuẩn bị
Xác nhận file `result/sft_train_ready.jsonl` có đủ 91 mẫu với Recency Anchor và code chuẩn.

In [ ]:
import json

sft_file = 'result/sft_train_ready.jsonl'
assert os.path.exists(sft_file), f'Không tìm thấy {sft_file}'

with open(sft_file, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

print(f'✅ Tổng số mẫu SFT: {len(records)}')
print(f'✅ Mẫu đầu tiên (ID: {records[0]["id"]} | Shop: {records[0]["business_id"]}):')
print('--- System Prompt (đầu):', records[0]['messages'][0]['content'][:120], '...')
print('--- User Prompt (đuôi có Recency Anchor):', repr(records[0]['messages'][1]['content'][-250:]))
print('--- Assistant Code:
', records[0]['messages'][2]['content'][:300], '...')

## Bước 5: Huấn luyện SFT LoRA với Qwen2.5-Coder-7B-Instruct
Chạy script `scripts/train_sft_lora.py`:
- **Base model:** `Qwen/Qwen2.5-Coder-7B-Instruct` (hoặc dùng local cache `/content/models/Qwen2.5-Coder-7B-Instruct` nếu có)
- **LoRA Config:** `r=16`, `alpha=32`, `target_modules=all-linear`
- **Loss:** Completion-only (mask prompt bằng -100)
- **Epochs:** 3.0
- **Max seq len:** 12288 tokens
- **Learning rate:** 2e-4 với Cosine decay

In [ ]:
import subprocess, sys, time
from pathlib import Path
import shutil
from collections import deque

t_start = time.time()

# Use the updated script copied to Drive; fail if it cannot be synchronized.
shutil.copy2(Path(gdrive_project) / 'scripts/train_sft_lora.py', 'scripts/train_sft_lora.py')
base_path = Path('/content/models/Qwen2.5-Coder-7B-Instruct')
base_model = str(base_path) if base_path.exists() else 'Qwen/Qwen2.5-Coder-7B-Instruct'

command = [
    sys.executable, '-u', 'scripts/train_sft_lora.py',
    '--model', base_model,
    '--data', 'result/sft_train_ready.jsonl',
    '--output_dir', 'result/sft_adapter_qwen7b',
    '--max_seq_len', '12288',
    '--epochs', '3.0',
    '--batch_size', '1',
    '--grad_accum', '4',
    '--lr', '2e-4',
    '--lora_r', '16',
    '--lora_alpha', '32',
]

log_path = Path('result/sft_adapter_qwen7b/train.log')
log_path.parent.mkdir(parents=True, exist_ok=True)
tail = deque(maxlen=40)
print(f'Training log: {log_path.resolve()}', flush=True)
with log_path.open('w', encoding='utf-8') as log_file:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log_file.write(line)
                log_file.flush()
                tail.append(line)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
if return_code:
    raise RuntimeError(
        f'Training failed (exit {return_code}). Full log: {log_path.resolve()}\n'
        + ''.join(tail)
    )

duration = time.time() - t_start
print(f'🎉 Huấn luyện hoàn thành sau {duration/60:.2f} phút!')


## Bước 6: Kiểm tra Adapter đã lưu
Xác nhận các file trọng số LoRA (`adapter_model.safetensors`, `adapter_config.json`) đã được tạo thành công.

In [ ]:
!ls -la result/sft_adapter_qwen7b/final_adapter

## Bước 7: Đánh giá Pilot 30 câu với LoRA Adapter
Cấu hình file `configs/baseline_7b_a100.yaml` để nạp `adapter_path: result/sft_adapter_qwen7b/final_adapter`, chạy suy luận 30 câu và chấm điểm Penalized Accuracy so sánh với trước khi SFT (trước đó là PA = -0.4333).

In [ ]:
import json, subprocess, sys
from collections import deque
from pathlib import Path
import yaml
from huggingface_hub import hf_hub_download

adapter_path = Path('result/sft_adapter_qwen7b/final_adapter').resolve()
with (adapter_path / 'adapter_config.json').open(encoding='utf-8') as f:
    adapter_config = json.load(f)
base_reference = adapter_config['base_model_name_or_path']
if Path(base_reference).is_dir():
    model_path = Path(base_reference).resolve()
elif Path('/content/models/Qwen2.5-Coder-7B-Instruct/config.json').is_file():
    model_path = Path('/content/models/Qwen2.5-Coder-7B-Instruct')
else:
    # Locate a cached runtime file without requiring repository documentation.
    cached_config = Path(hf_hub_download(repo_id=base_reference, filename='config.json', local_files_only=True))
    # Materialize cache symlinks: the offline backend requires shards inside its model directory.
    import os, shutil
    def link_or_copy(source, target):
        source = Path(source).resolve()
        try:
            os.link(source, target)
        except OSError:
            shutil.copy2(source, target)
        return target
    model_path = Path('tmp/sft_base_model').resolve()
    shutil.copytree(cached_config.parent, model_path, dirs_exist_ok=True, copy_function=link_or_copy)

with Path('configs/baseline_7b_a100.yaml').open(encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['model']['local_model_dir'] = str(model_path)
cfg['model']['backend'] = 'transformers'
cfg['model']['adapter_path'] = str(adapter_path)
cfg['paths']['output_file'] = 'result/sft_7b_pilot/submission.jsonl'
cfg['paths']['logs_dir'] = 'result/sft_7b_pilot/logs'
cfg['paths']['log_file'] = 'result/sft_7b_pilot/run.log'

# Keep the baseline configuration separate from the adapter evaluation.
cfg_path = Path('result/sft_7b_pilot/config.yaml')
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
print(f'Base weights: {model_path}', flush=True)
print(f'Adapter: {adapter_path}', flush=True)

commands = [
    [sys.executable, '-u', 'scripts/run_inference.py',
     '--config', str(cfg_path),
     '--questions-file', 'result/baseline_7b_pilot/pilot_questions_30.jsonl'],
    [sys.executable, '-u', 'scripts/evaluate_local.py',
     '--pred', 'result/sft_7b_pilot/submission.jsonl',
     '--gold', 'eval_gold/pilot_gold_30.jsonl'],
]
for command in commands:
    tail = deque(maxlen=40)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if return_code:
        raise RuntimeError(f'{command[2]} failed (exit {return_code}).\n' + ''.join(tail))


## Bước 8: Đồng bộ Adapter & Kết quả về Google Drive
Lưu adapter và kết quả đánh giá vĩnh viễn trên Google Drive để đồng bộ với máy local.

In [ ]:
drive_adapter_dir = f'{gdrive_project}/result/sft_adapter_qwen7b'
!mkdir -p "$drive_adapter_dir"
!cp -r result/sft_adapter_qwen7b/* "$drive_adapter_dir/"

drive_pilot_dir = f'{gdrive_project}/result/sft_7b_pilot'
!mkdir -p "$drive_pilot_dir"
!cp -r result/sft_7b_pilot/* "$drive_pilot_dir/"

print(f'✅ Đã lưu toàn bộ LoRA Adapter về Google Drive: {drive_adapter_dir}')
print(f'✅ Đã lưu kết quả Pilot về Google Drive: {drive_pilot_dir}')

## Bước 9: Tự động Sàng lọc & Mở rộng Dataset SFT (Self-Training / STaR)
> **Chiến lược:** Tận dụng chính Qwen 7B để sinh code giải các câu hỏi trong tập train (`data/train/questions.jsonl`), sau đó đưa qua **Hệ thống lọc 4 Cổng khắt khe**:
> 1. **Cổng 1:** Chấm điểm thực tế (+1.0 điểm chuẩn BTC, khớp 100% với `answers.jsonl`).
> 2. **Cổng 2:** AST Data Access Check (`is_safe_code`): Bắt buộc code phải đọc dữ liệu thật (`read_csv`, `read_excel`, `sqlite3`), không được đoán mò hoặc in sẵn đáp án.
> 3. **Cổng 3:** Sandbox Execution: Code thực thi exit code = 0, JSON format chuẩn.
> 4. **Cổng 4:** Deduplication & Chuẩn hóa ChatML kèm Recency Anchor.
>
> Mọi mẫu code đạt chuẩn sẽ tự động gộp vào `result/sft_train_ready.jsonl` để tiếp tục fine-tune vòng tiếp theo!

In [ ]:
# Đồng bộ script sàng lọc mới nhất từ Google Drive
!cp "$gdrive_project/scripts/mine_and_expand_sft.py" scripts/mine_and_expand_sft.py

# 1. Sàng lọc ngay các câu làm đúng từ đợt chạy Pilot vừa qua
!python scripts/mine_and_expand_sft.py --scan-logs result/sft_7b_pilot/logs


In [ ]:
# 2. Tùy chọn: Chạy vLLM sinh code quy mô lớn cho 150 câu hỏi mới trong data/train và tự động sàng lọc
# (Tự động lọc +1đ, kiểm tra AST đọc file thật và gộp vào dataset)
!pip install -q vllm

!python scripts/mine_and_expand_sft.py \
    --rollout-vllm \
    --model Qwen/Qwen2.5-Coder-7B-Instruct \
    --limit 150 \
    --batch-size 16
